# 🤗 x 🦾: Training SmolVLA with LeRobot Notebook

Welcome to the **LeRobot SmolVLA training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `SmolVLA` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `SmolVLA` policy for 20,000 steps typically takes **about 5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer!

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


## Install conda
This cell uses `condacolab` to bootstrap a full Conda environment inside Google Colab.


In [2]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg (version 7.1.1), and installs the package in editable mode.


In [3]:
!git clone https://github.com/huggingface/lerobot.git
!conda install ffmpeg=7.1.1 -c conda-forge
!cd lerobot && pip install -e .

fatal: destination path 'lerobot' already exists and is not an empty directory.
Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ | / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 25.11.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.

Obtaining file:///content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-80.9.0-py3-none-any.whl (1.2 MB)
  Building editable for lerobot (pyproject.toml) ... done
  Created wheel for lerobot: filename=lerobot-0.4.3-0.editable-py3-none-any.whl size=12265 sha256=aa50b12d1889f6dd52d5e0968ee2344e99afcb136104caf19a0fae76663091ce
  St

## Weights & Biases login
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging.

In [4]:
!pip install --upgrade "wandb>=0.22.3"
!wandb login

  Using cached wandb-0.23.1-py3-none-manylinux_2_28_x86_64.whl.metadata (12 kB)
Using cached wandb-0.23.1-py3-none-manylinux_2_28_x86_64.whl (22.9 MB)
  Attempting uninstall: wandb
    Found existing installation: wandb 0.21.4
    Uninstalling wandb-0.21.4:
      Successfully uninstalled wandb-0.21.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lerobot 0.4.3 requires wandb<0.22.0,>=0.20.0, but you have wandb 0.23.1 which is incompatible.
wandb: Currently logged in as: crellian123 (crellian123-university-of-colorado-boulder) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Install SmolVLA dependencies

In [5]:
!cd lerobot && pip install -e ".[smolvla]"

Obtaining file:///content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached wandb-0.21.4-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
Using cached wandb-0.21.4-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.6 MB)
  Building editable for lerobot (pyproject.toml) ... done
  Created wheel for lerobot: filename=lerobot-0.4.3-0.editable-py3-none-any.whl size=12265 sha256=9d3f8cea8681692e8fa514a7927164aa401d75ffbec8f8f30cde39bbcdbcd157
  Stored in directory: /tmp/pip-ephem-wheel-cache-_zpn3x45/wheels/15/0d/02/b9c6ff1c78574dee99101ad231194b3425eb4cd784ce8c8338
Successfully built lerobot
  Attempting uninstall: wandb
    Found existing installation: wandb 0.23.1
    Uninstalling wandb-0.23.1:
      Successfully uninstalled wandb-0.23.1
  Attempting uninstall: lerobot
    Foun

In [15]:
# Install robomimic without dependencies, then install its other deps manually
!pip install robomimic --no-deps --no-cache-dir

# Install robomimic's other dependencies (excluding egl_probe)
!pip install h5py tensorboard tensorboardX imageio-ffmpeg --no-cache-dir

# Now install lerobot, skipping the egl_probe requirement
!cd lerobot && pip install --no-cache-dir -e ".[libero]" --no-build-isolation 2>&1 | grep -v "egl_probe" || true

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 230.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 413.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 437.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 236.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
robomimic 0.3.0 requires egl_probe>=1.0.1, which is not installed.
robomimic 0.3.0 requires matplotlib, which is not installed.
Obtaining file:///content/lerobot
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 31.4 MB/s eta 0:00:0

In [21]:
import os
import tarfile
import urllib.request
import shutil

# Clean up and create fresh directory
shutil.rmtree('/tmp/robomimic_patch', ignore_errors=True)
os.makedirs('/tmp/robomimic_patch', exist_ok=True)

# Download robomimic
url = "https://files.pythonhosted.org/packages/source/r/robomimic/robomimic-0.2.0.tar.gz"
tarball = "/tmp/robomimic_patch/robomimic-0.2.0.tar.gz"
urllib.request.urlretrieve(url, tarball)

# Extract
with tarfile.open(tarball, 'r:gz') as tar:
    tar.extractall('/tmp/robomimic_patch')

# Read and patch setup.py to remove egl_probe
setup_path = '/tmp/robomimic_patch/robomimic-0.2.0/setup.py'
with open(setup_path, 'r') as f:
    content = f.read()

# Remove the egl_probe dependency
content = content.replace('"egl_probe>=1.0.1",', '')
content = content.replace("'egl_probe>=1.0.1',", '')

with open(setup_path, 'w') as f:
    f.write(content)

print("Patched setup.py - egl_probe removed")

Patched setup.py - egl_probe removed


/tmp/ipython-input-1927019049.py:17: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('/tmp/robomimic_patch')


In [22]:
# Install the patched robomimic
!pip install /tmp/robomimic_patch/robomimic-0.2.0 --no-cache-dir

Processing /tmp/robomimic_patch/robomimic-0.2.0
  Preparing metadata (setup.py) ... done
  Created wheel for robomimic: filename=robomimic-0.2.0-py3-none-any.whl size=223300 sha256=4ec933af9c5c6873930364e81ce35336757d76098c478b209e3c69e69f146207
  Stored in directory: /tmp/pip-ephem-wheel-cache-a3j6o8s5/wheels/fa/0a/3f/25c843b8fc586bb5a4ff2817ac0773ebb081c04bfb52ab8050
Successfully built robomimic
  Attempting uninstall: robomimic
    Found existing installation: robomimic 0.3.0
    Uninstalling robomimic-0.3.0:
      Successfully uninstalled robomimic-0.3.0


In [23]:
# Now install lerobot with libero
!cd lerobot && pip install --no-cache-dir -e ".[libero]"

Obtaining file:///content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 187.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 MB 137.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 117.9 MB/s e

In [26]:
!MPLBACKEND=agg lerobot-eval \
  --policy.path="HuggingFaceVLA/smolvla_libero" \
  --env.type=libero \
  --env.task=libero_object \
  --eval.batch_size=2 \
  --eval.n_episodes=3

Streaming output truncated to the last 5000 lines.
Running rollout with at most 280 steps:  24% 68/280 [01:09<03:35,  1.02s/it, running_success_rate=0.0%]
Running rollout with at most 280 steps:  25% 69/280 [01:09<03:34,  1.02s/it, running_success_rate=0.0%]
Running rollout with at most 280 steps:  25% 69/280 [01:10<03:34,  1.02s/it, running_success_rate=0.0%]
Running rollout with at most 280 steps:  25% 70/280 [01:10<03:33,  1.02s/it, running_success_rate=0.0%]
Running rollout with at most 280 steps:  25% 70/280 [01:11<03:33,  1.02s/it, running_success_rate=0.0%]
Running rollout with at most 280 steps:  25% 71/280 [01:11<03:33,  1.02s/it, running_success_rate=0.0%]
Running rollout with at most 280 steps:  25% 71/280 [01:12<03:33,  1.02s/it, running_success_rate=0.0%]
Running rollout with at most 280 steps:  26% 72/280 [01:12<03:34,  1.03s/it, running_success_rate=0.0%]
Running rollout with at most 280 steps:  26% 72/280 [01:13<03:34,  1.03s/it, running_success_rate=0.0%]
Running rollo

In [29]:
from IPython.display import Video

output_video_path = "/content/outputs/eval/2026-01-08/21-25-40_libero_smolvla/videos/libero_object_1/eval_episode_0.mp4"
Video(output_video_path, embed=True)

## Start training SmolVLA with LeRobot

This cell runs the `train.py` script from the `lerobot` library to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--batch_size=64`: means the model processes 64 training samples in parallel before doing one gradient update. Reduce this number if you have a GPU with low memory.

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this.

In [ ]:
# Run this in a cell BEFORE your training command
import matplotlib
matplotlib.use('Agg')

# Then patch the environment for subprocesses
import os
os.environ['MPLBACKEND'] = 'Agg'

# Also unset the problematic inline backend if it exists
if 'MPLBACKEND' in os.environ and 'inline' in os.environ['MPLBACKEND']:
    os.environ['MPLBACKEND'] = 'Agg'

In [43]:
!cd lerobot && rm -rf ./outputs/trainv5

In [44]:
!cd lerobot && MPLBACKEND=agg lerobot-train \
  --policy.type=smolvla \
  --policy.repo_id=${HF_USER}/libero-test \
  --policy.load_vlm_weights=true \
  --dataset.repo_id=HuggingFaceVLA/libero \
  --env.type=libero \
  --env.task=libero_10 \
  --policy.device=cuda \
  --wandb.enable=true \
  --output_dir=./outputs/trainv5/libero_smolvla_scratch \
  --job_name=libero_smolvla_scratch \
  --steps=100000 \
  --batch_size=4 \
  --eval.batch_size=1 \
  --eval.n_episodes=1 \
  --eval_freq=10000

Streaming output truncated to the last 5000 lines.
Running rollout with at most 520 steps:  23% 121/520 [00:20<01:05,  6.08it/s, running_success_rate=0.0%]
Running rollout with at most 520 steps:  23% 121/520 [00:20<01:05,  6.08it/s, running_success_rate=0.0%]
Running rollout with at most 520 steps:  23% 122/520 [00:20<01:04,  6.15it/s, running_success_rate=0.0%]
Running rollout with at most 520 steps:  23% 122/520 [00:20<01:04,  6.15it/s, running_success_rate=0.0%]
Running rollout with at most 520 steps:  24% 123/520 [00:20<01:03,  6.22it/s, running_success_rate=0.0%]
Running rollout with at most 520 steps:  24% 123/520 [00:20<01:03,  6.22it/s, running_success_rate=0.0%]
Running rollout with at most 520 steps:  24% 124/520 [00:20<01:03,  6.27it/s, running_success_rate=0.0%]
Running rollout with at most 520 steps:  24% 124/520 [00:20<01:03,  6.27it/s, running_success_rate=0.0%]
Running rollout with at most 520 steps:  24% 125/520 [00:20<01:03,  6.21it/s, running_success_rate=0.0%]
Runn

## Login into Hugging Face Hub
Now after training is done login into the Hugging Face hub and upload the last checkpoint

In [ ]:
!huggingface-cli login

In [ ]:
!huggingface-cli upload ${HF_USER}/my_smolvla \
  /content/lerobot/outputs/train/my_smolvla/checkpoints/last/pretrained_model